In [1]:
import numpy as np
import pandas as pd
import os
from glob import glob

# LIV experiment for 1 day and 4 days

### 1 day:
- 6x - 20 min vibrations per day, 1 hour in between
- fix 0, 1 and 3 hours
- starvation: 10% BCS and 2% BCS
- adipogenic: adipo and non-adipo media

### 4 dyas: 4x per day:
- 4x - 20 min vibrations per day, 1 hour in between
- fix 3 hours
- adipogenic: adipo and non-adipo media

## Data Loading

In [2]:
# Define the directory path
directory_path = r"Z:\Common\BMMB\LSM900 files\Uzer Lab\NN\2025.01.24_LIV_4_and1_day\Experiments_data\arivis_data\5x_whole_plate"

# Get a list of all Excel files in the directory
excel_files = glob(os.path.join(directory_path, "*.xlsx"))

# Initialize an empty list to store dataframes
df_list = []

# Read each Excel file and append to the list
for file in excel_files:
    try:
        df = pd.read_excel(file, sheet_name=None)  # Read all sheets
        for sheet_name, sheet_df in df.items():
            sheet_df["File_Name"] = os.path.basename(file)  # Add filename column
            sheet_df["Sheet_Name"] = sheet_name  # Add sheet name column
            df_list.append(sheet_df)
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Combine all DataFrames
if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
    print("Data successfully combined!")
else:
    combined_df = pd.DataFrame()
    print("No data found.")

combined_df


Data successfully combined!


,Name,"Sum, Intensities #1","Sum, Intensities #2","Area, 2D Oriented Bounds (µm²)",File_Name,Sheet_Name
0,Sphere #002,206378591732,67220360145,4.156862e+08,1D-CNT-1-01 (2)-features (4).xlsx,Objects
1,Segment #001 (Blob Finder),325323,250855,5.171237e+02,1D-CNT-1-01 (2)-features (4).xlsx,Objects
2,Segment #002 (Blob Finder),398001,229718,4.596662e+02,1D-CNT-1-01 (2)-features (4).xlsx,Objects
3,Segment #003 (Blob Finder),449946,235989,4.596629e+02,1D-CNT-1-01 (2)-features (4).xlsx,Objects
4,Segment #004 (Blob Finder),839688,276973,4.218185e+02,1D-CNT-1-01 (2)-features (4).xlsx,Objects
...,...,...,...,...,...,...
1582261,Segment #52662 (Blob Finder),66386,58619,3.548237e+02,D4-LIV-8-01-features.xlsx,Objects
1582262,Segment #52663 (Blob Finder),182084,371182,1.135622e+03,D4-LIV-8-01-features.xlsx,Objects
1582263,Segment #52664 (Blob Finder),136462,232276,8.348963e+02,D4-LIV-8-01-features.xlsx,Objects
1582264,Segment #52665 (Blob Finder),101635,166021,6.101163e+02,D4-LIV-8-01-features.xlsx,Objects


In [3]:
combined_df.columns

Index(['Name', 'Sum, Intensities #1', 'Sum, Intensities #2',
       'Area, 2D Oriented Bounds (µm²)', 'File_Name', 'Sheet_Name'],
      dtype='object')

In [4]:
# Rename columns to shorter, more descriptive names
rename_dict = {
    'Name': 'Sample_Name',
    'Sum, Intensities #1': 'Intensity_1',
    'Sum, Intensities #2': 'Intensity_2',
    'Area, 2D Oriented Bounds (µm²)': 'Area_2D',
    'File_Name': 'File_Name'
}
combined_df = combined_df.rename(columns=rename_dict)

# Step 2: Remove 'Sheet_Name' column
combined_df = combined_df.drop(columns=['Sheet_Name'], errors='ignore')

# Step 3: Create 'Vibration' column based on 'File_Name'
combined_df['Vibration'] = combined_df['File_Name'].apply(lambda x: 'LIV' if 'LIV' in x else 'CNT' if 'CNT' in x else 'Unknown')

# Step 4: Create 'Days' column based on '1D' or '4D' in 'File_Name'
combined_df['Days'] = combined_df['File_Name'].apply(lambda x: 1 if '1D' in x else 4 if 'D4' in x else None)

# Step 5: Extract 'Cell Type' and 'Plate Number'
def extract_cell_type_and_plate(file_name):
    parts = file_name.split('-')

    # Extracting number after CNT or LIV
    try:
        exp_index = next(i for i, part in enumerate(parts) if part in ['CNT', 'LIV'])
        exp_number = parts[exp_index + 1]
    except (StopIteration, IndexError):
        exp_number = None

    return exp_number

combined_df[['Plate_Number']] = combined_df['File_Name'].apply(
    lambda x: pd.Series(extract_cell_type_and_plate(x))
)

In [5]:
combined_df

,Sample_Name,Intensity_1,Intensity_2,Area_2D,File_Name,Vibration,Days,Plate_Number
0,Sphere #002,206378591732,67220360145,4.156862e+08,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1
1,Segment #001 (Blob Finder),325323,250855,5.171237e+02,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1
2,Segment #002 (Blob Finder),398001,229718,4.596662e+02,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1
3,Segment #003 (Blob Finder),449946,235989,4.596629e+02,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1
4,Segment #004 (Blob Finder),839688,276973,4.218185e+02,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1
...,...,...,...,...,...,...,...,...
1582261,Segment #52662 (Blob Finder),66386,58619,3.548237e+02,D4-LIV-8-01-features.xlsx,LIV,4,8
1582262,Segment #52663 (Blob Finder),182084,371182,1.135622e+03,D4-LIV-8-01-features.xlsx,LIV,4,8
1582263,Segment #52664 (Blob Finder),136462,232276,8.348963e+02,D4-LIV-8-01-features.xlsx,LIV,4,8
1582264,Segment #52665 (Blob Finder),101635,166021,6.101163e+02,D4-LIV-8-01-features.xlsx,LIV,4,8


In [6]:
# Define lookup tables as dictionaries
lookup_1D = {
    1: {'Starving': False, 'Adipogenic_Media': False},
    2: {'Starving': False, 'Adipogenic_Media': False},
    3: {'Starving': False, 'Adipogenic_Media': False},
    4: {'Starving': True, 'Adipogenic_Media': False},
    5: {'Starving': True, 'Adipogenic_Media': False},
    6: {'Starving': True, 'Adipogenic_Media': False},
    7: {'Starving': False, 'Adipogenic_Media': True},
    8: {'Starving': False, 'Adipogenic_Media': True},
    9: {'Starving': False, 'Adipogenic_Media': True},
    10: {'Starving': True, 'Adipogenic_Media': True},
    11: {'Starving': True, 'Adipogenic_Media': True},
    12: {'Starving': True, 'Adipogenic_Media': True},
    13: {'Starving': False, 'Adipogenic_Media': False},
    14: {'Starving': False, 'Adipogenic_Media': False},
    15: {'Starving': False, 'Adipogenic_Media': False},
    16: {'Starving': True, 'Adipogenic_Media': False},
    17: {'Starving': True, 'Adipogenic_Media': False},
    18: {'Starving': True, 'Adipogenic_Media': False},
    19: {'Starving': False, 'Adipogenic_Media': True},
    20: {'Starving': False, 'Adipogenic_Media': True},
    21: {'Starving': False, 'Adipogenic_Media': True},
    22: {'Starving': True, 'Adipogenic_Media': True},
    23: {'Starving': True, 'Adipogenic_Media': True},
    24: {'Starving': True, 'Adipogenic_Media': True},
}

lookup_4D = {
    1: {'Starving': False, 'Adipogenic_Media': False},
    2: {'Starving': False, 'Adipogenic_Media': False},
    3: {'Starving': False, 'Adipogenic_Media': True},
    4: {'Starving': False, 'Adipogenic_Media': True},
    5: {'Starving': False, 'Adipogenic_Media': False},
    6: {'Starving': False, 'Adipogenic_Media': False},
    7: {'Starving': False, 'Adipogenic_Media': True},
    8: {'Starving': False, 'Adipogenic_Media': True},
}

# Function to map values
def map_conditions(row):
    if row['Days'] == 1:
        return lookup_1D.get(row['Plate_Number'], {'Starving': None, 'Adipogenic_Media': None})
    elif row['Days'] == 4:
        return lookup_4D.get(row['Plate_Number'], {'Starving': None, 'Adipogenic_Media': None})
    else:
        return {'Starving': None, 'Adipogenic_Media': None}

# Apply mapping function
conditions = combined_df.apply(map_conditions, axis=1)

# Convert mapping output into new DataFrame columns
combined_df['Starving'] = conditions.apply(lambda x: x['Starving'])
combined_df['Adipogenic_Media'] = conditions.apply(lambda x: x['Adipogenic_Media'])

In [7]:
combined_df

,Sample_Name,Intensity_1,Intensity_2,Area_2D,File_Name,Vibration,Days,Plate_Number,Starving,Adipogenic_Media
0,Sphere #002,206378591732,67220360145,4.156862e+08,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1,None,None
1,Segment #001 (Blob Finder),325323,250855,5.171237e+02,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1,None,None
2,Segment #002 (Blob Finder),398001,229718,4.596662e+02,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1,None,None
3,Segment #003 (Blob Finder),449946,235989,4.596629e+02,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1,None,None
4,Segment #004 (Blob Finder),839688,276973,4.218185e+02,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1,None,None
...,...,...,...,...,...,...,...,...,...,...
1582261,Segment #52662 (Blob Finder),66386,58619,3.548237e+02,D4-LIV-8-01-features.xlsx,LIV,4,8,None,None
1582262,Segment #52663 (Blob Finder),182084,371182,1.135622e+03,D4-LIV-8-01-features.xlsx,LIV,4,8,None,None
1582263,Segment #52664 (Blob Finder),136462,232276,8.348963e+02,D4-LIV-8-01-features.xlsx,LIV,4,8,None,None
1582264,Segment #52665 (Blob Finder),101635,166021,6.101163e+02,D4-LIV-8-01-features.xlsx,LIV,4,8,None,None


In [8]:
import numpy as np
import pandas as pd
import os
from glob import glob

# Define directory path
directory_path = r"Z:\Common\BMMB\LSM900 files\Uzer Lab\NN\2025.01.24_LIV_4_and1_day\Experiments_data\arivis_data\5x_whole_plate"

# Get list of all Excel files in the directory
excel_files = glob(os.path.join(directory_path, "*.xlsx"))

# Function to read all sheets from an Excel file
def read_excel_sheets(file_path):
    try:
        df_dict = pd.read_excel(file_path, sheet_name=None)  # Read all sheets
        df_list = []
        file_name = os.path.basename(file_path)

        for sheet_name, df in df_dict.items():
            df['File_Name'] = file_name
            df_list.append(df)

        return pd.concat(df_list, ignore_index=True)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()

# Read all Excel files in parallel using list comprehension
df_list = [read_excel_sheets(file) for file in excel_files]

# Combine all DataFrames
combined_df = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()

# Rename columns
rename_dict = {
    'Name': 'Sample_Name',
    'Sum, Intensities #1': 'Intensity_1',
    'Sum, Intensities #2': 'Intensity_2',
    'Area, 2D Oriented Bounds (µm²)': 'Area_2D'
}
combined_df.rename(columns=rename_dict, inplace=True)

# Remove 'Sheet_Name' column if present
combined_df.drop(columns=['Sheet_Name'], errors='ignore', inplace=True)

# Create 'Vibration' column based on 'File_Name'
combined_df['Vibration'] = np.where(combined_df['File_Name'].str.contains('LIV'), 'LIV',
                                    np.where(combined_df['File_Name'].str.contains('CNT'), 'CNT', 'Unknown'))

# Create 'Days' column based on '1D' or '4D' in 'File_Name'
combined_df['Days'] = np.where(combined_df['File_Name'].str.contains('1D'), 1,
                               np.where(combined_df['File_Name'].str.contains('D4'), 4, None))

# Extract 'Plate_Number' from 'File_Name'
combined_df['Plate_Number'] = combined_df['File_Name'].str.extract(r'-(\d+)-').astype(float)

# Define lookup tables for conditions
lookup_conditions = {
    1: {'Starving': False, 'Adipogenic_Media': False}, 2: {'Starving': False, 'Adipogenic_Media': False},
    3: {'Starving': False, 'Adipogenic_Media': False}, 4: {'Starving': True, 'Adipogenic_Media': False},
    5: {'Starving': True, 'Adipogenic_Media': False}, 6: {'Starving': True, 'Adipogenic_Media': False},
    7: {'Starving': False, 'Adipogenic_Media': True}, 8: {'Starving': False, 'Adipogenic_Media': True},
    9: {'Starving': False, 'Adipogenic_Media': True}, 10: {'Starving': True, 'Adipogenic_Media': True},
    11: {'Starving': True, 'Adipogenic_Media': True}, 12: {'Starving': True, 'Adipogenic_Media': True},
    13: {'Starving': False, 'Adipogenic_Media': False}, 14: {'Starving': False, 'Adipogenic_Media': False},
    15: {'Starving': False, 'Adipogenic_Media': False}, 16: {'Starving': True, 'Adipogenic_Media': False},
    17: {'Starving': True, 'Adipogenic_Media': False}, 18: {'Starving': True, 'Adipogenic_Media': False},
    19: {'Starving': False, 'Adipogenic_Media': True}, 20: {'Starving': False, 'Adipogenic_Media': True},
    21: {'Starving': False, 'Adipogenic_Media': True}, 22: {'Starving': True, 'Adipogenic_Media': True},
    23: {'Starving': True, 'Adipogenic_Media': True}, 24: {'Starving': True, 'Adipogenic_Media': True},
}

lookup_conditions_4D = {
    1: {'Starving': False, 'Adipogenic_Media': False}, 2: {'Starving': False, 'Adipogenic_Media': False},
    3: {'Starving': False, 'Adipogenic_Media': True}, 4: {'Starving': False, 'Adipogenic_Media': True},
    5: {'Starving': False, 'Adipogenic_Media': False}, 6: {'Starving': False, 'Adipogenic_Media': False},
    7: {'Starving': False, 'Adipogenic_Media': True}, 8: {'Starving': False, 'Adipogenic_Media': True},
}

# Apply lookup table mapping
def map_conditions(row):
    if row['Days'] == 1:
        return lookup_conditions.get(row['Plate_Number'], {'Starving': None, 'Adipogenic_Media': None})
    elif row['Days'] == 4:
        return lookup_conditions_4D.get(row['Plate_Number'], {'Starving': None, 'Adipogenic_Media': None})
    return {'Starving': None, 'Adipogenic_Media': None}

# Use vectorized operations instead of apply() for performance
conditions_df = pd.DataFrame([map_conditions(row) for _, row in combined_df.iterrows()])

# Merge the new columns into the main DataFrame
combined_df = pd.concat([combined_df, conditions_df], axis=1)
combined_df

,Sample_Name,Intensity_1,Intensity_2,Area_2D,File_Name,Vibration,Days,Plate_Number,Starving,Adipogenic_Media
0,Sphere #002,206378591732,67220360145,4.156862e+08,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1.0,False,False
1,Segment #001 (Blob Finder),325323,250855,5.171237e+02,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1.0,False,False
2,Segment #002 (Blob Finder),398001,229718,4.596662e+02,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1.0,False,False
3,Segment #003 (Blob Finder),449946,235989,4.596629e+02,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1.0,False,False
4,Segment #004 (Blob Finder),839688,276973,4.218185e+02,1D-CNT-1-01 (2)-features (4).xlsx,CNT,1,1.0,False,False
...,...,...,...,...,...,...,...,...,...,...
1582261,Segment #52662 (Blob Finder),66386,58619,3.548237e+02,D4-LIV-8-01-features.xlsx,LIV,4,8.0,False,True
1582262,Segment #52663 (Blob Finder),182084,371182,1.135622e+03,D4-LIV-8-01-features.xlsx,LIV,4,8.0,False,True
1582263,Segment #52664 (Blob Finder),136462,232276,8.348963e+02,D4-LIV-8-01-features.xlsx,LIV,4,8.0,False,True
1582264,Segment #52665 (Blob Finder),101635,166021,6.101163e+02,D4-LIV-8-01-features.xlsx,LIV,4,8.0,False,True


In [9]:
combined_df.columns

Index(['Sample_Name', 'Intensity_1', 'Intensity_2', 'Area_2D', 'File_Name',
       'Vibration', 'Days', 'Plate_Number', 'Starving', 'Adipogenic_Media'],
      dtype='object')

In [10]:
# Ensure column name is correct by checking available columns
actual_columns = [col.strip().lower() for col in combined_df.columns]
expected_column = "sample_name"

# Standardize column names
if expected_column not in actual_columns:
    raise KeyError(f"Column '{expected_column}' not found. Available columns: {combined_df.columns}")

# Step 1: Create nuclei_df with rows where "Sample_Name" contains "Segment"
nuclei_df = combined_df[combined_df['Sample_Name'].str.contains("Segment", na=False, case=False)]

# Step 2: Create plate_df with rows where "Sample_Name" contains "Sphere"
plate_df = combined_df[combined_df['Sample_Name'].str.contains("Sphere", na=False, case=False)]
plate_df = plate_df[(plate_df['Intensity_1'] > 100)]

# Step 3: Create filtered_nuclei_df where 100 < Area_2D < 3000
filtered_nuclei_df = nuclei_df[(nuclei_df['Area_2D'] > 100) & (nuclei_df['Area_2D'] < 3000)]

# Step 4: Group filtered_nuclei_df by 'Vibration', 'Days', 'Plate_Number' and calculate mean for numerical columns
# Add a column showing the count of rows per group
row_counts = filtered_nuclei_df.groupby(['Vibration', 'Days', 'Plate_Number']).size().reset_index(name='Row_Count')

# Group and sum numeric values, then merge row counts
grouped_filtered_nuclei = filtered_nuclei_df.groupby(['Vibration', 'Days', 'Plate_Number']).sum(numeric_only=True).reset_index()
grouped_filtered_nuclei = grouped_filtered_nuclei.merge(row_counts, on=['Vibration', 'Days', 'Plate_Number'], how='left')


In [11]:
grouped_filtered_nuclei

,Vibration,Days,Plate_Number,Intensity_1,Intensity_2,Area_2D,Starving,Adipogenic_Media,Row_Count
0,CNT,1,1.0,12745026599,10366034425,1.855144e+07,0,0,30615
1,CNT,1,2.0,19203633393,12172128451,2.715463e+07,0,0,35264
2,CNT,1,3.0,24395333717,15348446683,3.166318e+07,0,0,43296
3,CNT,1,4.0,15298553758,11540982623,2.217829e+07,35869,0,35869
4,CNT,1,5.0,17558698254,11383673978,2.568066e+07,35640,0,35640
5,CNT,1,6.0,17919659293,12646441170,2.896699e+07,40161,0,40161
6,CNT,1,7.0,17666055351,13216567105,2.562245e+07,0,35635,35635
7,CNT,1,8.0,16829609089,11953618385,2.745813e+07,0,36902,36902
8,CNT,1,9.0,15899385323,11527387811,2.663380e+07,0,34968,34968
9,CNT,1,10.0,8917235447,8260043812,1.810241e+07,26556,26556,26556


In [12]:
plate_df.to_csv('plate.csv', index=False)
grouped_filtered_nuclei.to_csv('grouped_filtered_nuclei.csv', index=False)
filtered_nuclei_df.to_csv('filtered_nuclei.csv', index=False)
nuclei_df.to_csv('nuclei.csv', index=False)



In [13]:
import os

# Ensure filtered_nuclei_df exists before proceeding
if 'filtered_nuclei_df' not in locals():
    raise NameError("filtered_nuclei_df is not defined. Ensure it is created before running this code.")

# Define base directory for saving files
save_directory = "filtered_nuclei_data"
os.makedirs(save_directory, exist_ok=True)

# Save separate CSV files for each Plate_Number
for plate_num, df_subset in filtered_nuclei_df.groupby('Plate_Number'):
    file_path = os.path.join(save_directory, f"filtered_nuclei_plate_{int(plate_num)}.csv")
    df_subset.to_csv(file_path, index=False)

# Display a message confirming the files have been saved
print(f"Filtered nuclei data saved as separate CSV files in the directory: {save_directory}")


Filtered nuclei data saved as separate CSV files in the directory: filtered_nuclei_data


In [14]:
# Define base directory for saving files
save_directory = "filtered_nuclei_data"
os.makedirs(save_directory, exist_ok=True)

# Group by sets of three plates (1-3, 4-6, etc.)
filtered_nuclei_df['Plate_Group'] = ((filtered_nuclei_df['Plate_Number'] - 1) // 3) + 1

for group_num, df_subset in filtered_nuclei_df.groupby('Plate_Group'):
    file_path = os.path.join(save_directory, f"filtered_nuclei_plates_{(group_num-1)*3+1}_to_{group_num*3}.csv")
    df_subset.to_csv(file_path, index=False)

C:\Users\nnina\AppData\Local\Temp\ipykernel_25100\2712392821.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_nuclei_df['Plate_Group'] = ((filtered_nuclei_df['Plate_Number'] - 1) // 3) + 1
